In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [7]:
import pandas as pd
import numpy as np

# Đường dẫn chính xác đã được xác thực từ kết quả quét file
train_path = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv'
test_path = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv'

# Đọc dữ liệu
train_hp = pd.read_csv(train_path)
test_hp = pd.read_csv(test_path)

print("Kích thước tập Train giá nhà:", train_hp.shape)
print("Kích thước tập Test giá nhà:", test_hp.shape)

# Kiểm tra các cột bị thiếu dữ liệu 
missing_count = train_hp.isnull().sum()
missing_percent = (train_hp.isnull().sum() / len(train_hp)) * 100

# Tạo bảng tổng hợp các cột bị thiếu dữ liệu
missing_df = pd.DataFrame({
    'Số lượng thiếu': missing_count,
    'Tỷ lệ phần trăm (%)': missing_percent
})

# Lọc ra các cột thực sự có giá trị thiếu và sắp xếp theo số lượng giảm dần
missing_df = missing_df[missing_df['Số lượng thiếu'] > 0].sort_values(by='Số lượng thiếu', ascending=False)

print("\nTop các cột bị thiếu dữ liệu trong tập Train:")
display(missing_df.head(15))

Kích thước tập Train giá nhà: (1460, 81)
Kích thước tập Test giá nhà: (1459, 80)

Top các cột bị thiếu dữ liệu trong tập Train:


,Số lượng thiếu,Tỷ lệ phần trăm (%)
PoolQC,1453,99.520548
MiscFeature,1406,96.301370
Alley,1369,93.767123
Fence,1179,80.753425
MasVnrType,872,59.726027
FireplaceQu,690,47.260274
LotFrontage,259,17.739726
GarageType,81,5.547945
GarageYrBlt,81,5.547945
GarageFinish,81,5.547945


In [8]:
# 1. Các cột dạng chữ (categorical) mà giá trị thiếu có nghĩa là "Không có" (None)
cols_cat_none = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType'
]

# Điền giá trị 'None' cho các cột này ở cả tập Train và Test
for col in cols_cat_none:
    if col in train_hp.columns:
        train_hp[col] = train_hp[col].fillna('None')
    if col in test_hp.columns:
        test_hp[col] = test_hp[col].fillna('None')

# 2. Các cột dạng số (numerical) liên quan đến Gara hoặc diện tích mà thiếu nghĩa là 0 (Không có)
cols_num_zero = ['GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']

for col in cols_num_zero:
    if col in train_hp.columns:
        train_hp[col] = train_hp[col].fillna(0)
    if col in test_hp.columns:
        test_hp[col] = test_hp[col].fillna(0)

# 3. Riêng cột LotFrontage (Đường diện tiền nhà) thiếu thì ta điền bằng số trung median của cột
if 'LotFrontage' in train_hp.columns:
    median_lot = train_hp['LotFrontage'].median()
    train_hp['LotFrontage'] = train_hp['LotFrontage'].fillna(median_lot)
    test_hp['LotFrontage'] = test_hp['LotFrontage'].fillna(median_lot)

# Kiểm tra lại xem còn cột nào bị thiếu dữ liệu trong tập train không
con_thieu_train = train_hp.isnull().sum().sum()
con_thieu_test = test_hp.isnull().sum().sum()

print(f"Tổng số ô còn thiếu dữ liệu trong tập Train: {con_thieu_train}")
print(f"Tổng số ô còn thiếu dữ liệu trong tập Test: {con_thieu_test}")

Tổng số ô còn thiếu dữ liệu trong tập Train: 1
Tổng số ô còn thiếu dữ liệu trong tập Test: 12


In [9]:
# 1. Điền nốt các cột số còn sót lại bằng số trung vị (median) của tập train
cols_num_train = train_hp.select_dtypes(include=['int64', 'float64']).columns
for col in cols_num_train:
    if train_hp[col].isnull().sum() > 0:
        median_val = train_hp[col].median()
        train_hp[col] = train_hp[col].fillna(median_val)
        
cols_num_test = test_hp.select_dtypes(include=['int64', 'float64']).columns
for col in cols_num_test:
    if test_hp[col].isnull().sum() > 0:
        median_val = train_hp[col].median() # Lấy median từ tập train để điền cho test (tránh rò rỉ dữ liệu)
        test_hp[col] = test_hp[col].fillna(median_val)

# 2. Điền nốt các cột chữ (categorical) còn sót lại bằng giá trị xuất hiện nhiều nhất (mode)
cols_cat_train = train_hp.select_dtypes(include=['object']).columns
for col in cols_cat_train:
    if train_hp[col].isnull().sum() > 0:
        mode_val = train_hp[col].mode()[0]
        train_hp[col] = train_hp[col].fillna(mode_val)

cols_cat_test = test_hp.select_dtypes(include=['object']).columns
for col in cols_cat_test:
    if test_hp[col].isnull().sum() > 0:
        mode_val = train_hp[col].mode()[0] # Lấy mode từ tập train để điền cho test
        test_hp[col] = test_hp[col].fillna(mode_val)

# 3. Kiểm tra lại lần cuối xem còn ô trống nào không
print("Tổng số ô còn thiếu sau khi xử lý triệt để - Train:", train_hp.isnull().sum().sum())
print("Tổng số ô còn thiếu sau khi xử lý triệt để - Test:", test_hp.isnull().sum().sum())

Tổng số ô còn thiếu sau khi xử lý triệt để - Train: 0
Tổng số ô còn thiếu sau khi xử lý triệt để - Test: 0
